In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import re
import joblib
import pandas as pd
import numpy as np
import clingo
from collections import defaultdict

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.path.abspath(os.getcwd()))

from utils.asp_utils import generate_board_baseline_facts, generate_ml_link_facts, generate_agenda_facts
from utils.data_utils import generate_rich_link_dataset

In [ ]:
def predict_daily_affinities(json_path, target_date, org_name, output_ml_facts="encodings/facts/ml_link_facts.lp"):
    """
    Carica il modello specifico per l'ospedale, genera le combinazioni per la giornata
    e restituisce i fatti probabilistici.
    """
    model_path = f"saved_models/tabpfn_link_predictor_{org_name}.pkl"
    cols_path = f"saved_models/train_columns_{org_name}.pkl"
    
    try:
        classifier = joblib.load(model_path)
        train_columns = joblib.load(cols_path)
    except Exception as e:
        print(f"[!] Errore: Impossibile caricare il modello per {org_name}. ({e})")
        return False, None

    # Generazione Dataset
    df_day = generate_rich_link_dataset(json_path, negative_ratio=100, random_seed=42)
    df_today = df_day[df_day['planning_date'].astype(str).str.contains(target_date)].copy()
    
    if df_today.empty:
        return False, None
        
    # Preprocessing Identico al Training
    cat_cols = df_today.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
    cols_to_encode = [c for c in cat_cols if c not in ['operator_id', 'patient_id']]
    df_encoded = pd.get_dummies(df_today, columns=cols_to_encode, drop_first=True, dtype=int)
    
    # Allineamento rigoroso delle colonne
    for col in train_columns:
        if col not in df_encoded.columns:
            df_encoded[col] = 0
    df_encoded = df_encoded[train_columns]
    
    # Predizione
    X_np = df_encoded.values.astype(np.float32)
    y_pred_proba = classifier.predict_proba(X_np)[:, 1]
    
    df_today['Probability'] = y_pred_proba
    df_today['Patient'] = df_today['patient_id']
    df_today['Operator'] = df_today['operator_id']
    
    # Generazione Fatti ASP
    generate_ml_link_facts(df_today, output_ml_facts)
    return True, df_today

In [ ]:
def analizza_continuita_assistenziale(df_assignments, df_day_dataset):
    """
    Confronta gli assegnamenti del solver con il dataset di partenza per capire
    quanti pazienti sono stati assegnati a un medico che conoscevano già.
    """
    storici = 0
    nuovi = 0
    
    # Creiamo un dizionario veloce dal dataset: (pat, op) -> historical_count
    storia_diz = {}
    for _, row in df_day_dataset.iterrows():
        storia_diz[(row['patient_id'], row['operator_id'])] = row['historical_pair_count']
        
    for ass in df_assignments:
        pat = ass['patient_id']
        op = ass['operator_id']
        
        if op == -1:
            continue # Ignoriamo i pazienti a terra
            
        volte_visto_in_passato = storia_diz.get((pat, op), 0)
        
        if volte_visto_in_passato > 0:
            storici += 1
        else:
            nuovi += 1
            
    totale = storici + nuovi
    perc_storici = (storici / totale) * 100 if totale > 0 else 0
    
    print(f"Pazienti totali assegnati validi: {totale}")
    print(f"-> Assegnati a medici STORICI (Continuità): {storici} ({perc_storici:.1f}%)")
    print(f"-> Assegnati a medici NUOVI: {nuovi} ({100 - perc_storici:.1f}%)")
    return perc_storici

In [ ]:
def run_board_solver(rules_file, facts_file, ml_file=None, timeout=20.0):
    ctl = clingo.Control(["--opt-strategy=usc,k,0,4", "--heuristic=Domain"])
    ctl.load(rules_file)
    ctl.load(facts_file)
    if ml_file:
        ctl.load(ml_file)
        
    ctl.ground([("base", [])])
    
    best_cost = None
    board_assignments = []
    
    def on_model(model):
        nonlocal best_cost, board_assignments
        best_cost = model.cost
        raw_symbols = [str(sym) for sym in model.symbols(shown=True)]
        
        pattern = re.compile(r"assignment\((-?\d+),\s*(\d+),\s*(\d+),\s*(\d+)\)")
        temp_ass = []
        for sym in raw_symbols:
            match = pattern.search(sym)
            if match:
                temp_ass.append({
                    'operator_id': int(match.group(1)),
                    'patient_id': int(match.group(2))
                })
        board_assignments = temp_ass
        
    t0 = time.perf_counter()
    with ctl.solve(on_model=on_model, async_=True) as handle:
        finished = handle.wait(timeout)
        if not finished:
            handle.cancel()
            
    exec_time = time.perf_counter() - t0
    return board_assignments, exec_time, best_cost

def run_agenda_solver(agenda_facts_file, agenda_rules_file="encodings/agenda_rules.lp", timeout=30.0):
    ctl = clingo.Control(["--opt-mode=optN", "--parallel-mode=4"])
    ctl.load(agenda_rules_file)
    ctl.load(agenda_facts_file)
    ctl.ground([("base", [])])
    
    agenda_cost = []
    def on_model(model):
        nonlocal agenda_cost
        agenda_cost = model.cost
        
    t0 = time.perf_counter()
    with ctl.solve(on_model=on_model, async_=True) as handle:
        finished = handle.wait(timeout)
        if not finished:
            handle.cancel()
            
    exec_time = time.perf_counter() - t0
    
    # Costo a priorità 9 (La nostra valvola di sfogo per i pazienti lasciati a terra)
    dropped_sessions = agenda_cost[0] if agenda_cost and len(agenda_cost) == 10 else 0
    return exec_time, agenda_cost, dropped_sessions

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import clingo
import re

def parse_and_plot_agenda(facts_file, rules_file, title="Agenda Giornaliera"):
    print(f"\n--- Generazione Plot: {title} ---")
    ctl = clingo.Control(["--opt-mode=optN", "--parallel-mode=4"])
    ctl.load(rules_file)
    ctl.load(facts_file)
    ctl.ground([("base", [])])
    
    sessions = {}
    starts = {}
    lengths = {}
    
    # 1. Estrazione sicura dal file dei fatti
    with open(facts_file, 'r') as f:
        content = f.read()
        for match in re.finditer(r"session\(\s*(\d+),\s*(\d+),\s*(-?\d+)", content):
            sessions[int(match.group(1))] = {
                'pat': int(match.group(2)),
                'op': int(match.group(3))
            }
            
    print(f" -> Letti i fatti: Trovate {len(sessions)} sessioni.")

    modelli_trovati = 0

    def on_model(model):
        nonlocal starts, lengths, modelli_trovati
        modelli_trovati += 1
        print(f"    [OK] Trovato modello #{modelli_trovati} con costo: {model.cost}")
        
        starts_temp = {}
        lengths_temp = {}
        
        # Usiamo shown=True per catturare esattamente ciò che hai indicato con le direttive #show
        for sym in model.symbols(shown=True):
            if sym.name == "start":
                starts_temp[sym.arguments[0].number] = sym.arguments[2].number
            elif sym.name == "length":
                lengths_temp[sym.arguments[0].number] = sym.arguments[2].number
        
        if starts_temp:
            starts.clear()
            starts.update(starts_temp)
            lengths.clear()
            lengths.update(lengths_temp)

    print(" -> Risoluzione Agenda ASP in corso (max 60 secondi)...")
    
    with ctl.solve(on_model=on_model, async_=True) as handle:
        finished = handle.wait(60.0)
        if not finished:
            print("    [!] Timeout raggiunto. Fermo la ricerca e tengo il miglior modello.")
            handle.cancel()
        
        # Otteniamo l'esito finale REALE del solver
        esito = handle.get()
        print(f" -> Esito della ricerca: {esito}")
            
    print(f" -> Dati estratti finali: {len(starts)} orari di inizio e {len(lengths)} durate.")
    
    if modelli_trovati == 0:
        print("\n[!] ERRORE GRAVE: Il solutore ha restituito UNSATISFIABLE.")
        print("    Significa che le regole della Board hanno proposto assegnazioni che non c'entrano fisicamente negli orari.")
        return
        
    if not starts:
        print("\n[!] ERRORE: Modello trovato, ma le variabili 'start' e 'length' non sono state lette.")
        return

    # --- PLOTTING ---
    fig, ax = plt.subplots(figsize=(14, 8))
    ops_schedule = {}
    
    for sess_id, st in starts.items():
        if sess_id not in sessions or sess_id not in lengths: continue
        op = sessions[sess_id]['op']
        pat = sessions[sess_id]['pat']
        l = lengths[sess_id]
        
        if op not in ops_schedule:
            ops_schedule[op] = []
        
        start_hour = st / 6.0 
        duration_hour = l / 6.0
        ops_schedule[op].append((start_hour, duration_hour, pat))
        
    y_ticks = []
    y_labels = []
    
    for i, (op, blocks) in enumerate(sorted(ops_schedule.items())):
        y_pos = i * 10
        y_ticks.append(y_pos + 4)
        y_labels.append(f"OP {op}" if op != -1 else "A TERRA (-1)")
        
        for block in blocks:
            st_hr, dur_hr, pat = block
            facecolor = 'salmon' if op == -1 else 'skyblue'
            edgecolor = 'red' if op == -1 else 'royalblue'
            
            rect = patches.Rectangle((st_hr, y_pos), dur_hr, 8, facecolor=facecolor, edgecolor=edgecolor, alpha=0.8)
            ax.add_patch(rect)
            ax.text(st_hr + dur_hr/2, y_pos + 4, f"P {pat}", ha='center', va='center', fontsize=8, color='black', rotation=90 if dur_hr < 0.5 else 0)

    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)
    ax.set_xlabel("Orario della giornata (Ore, es. 8.0 = 08:00)")
    ax.set_title(title)
    ax.grid(True, axis='x', linestyle='--', alpha=0.6)
    
    ax.set_xlim(7, 20)
    plt.tight_layout()
    plt.show()

In [ ]:
def run_benchmark(json_path, start_date, end_date, timeout=20.0):
    dates = pd.date_range(start=start_date, end=end_date)
    results = []
    
    os.makedirs("results", exist_ok=True)
    os.makedirs("encodings/facts", exist_ok=True)
    
    # Rilevamento automatico dell'organizzazione
    if "Org6" in json_path:
        org_name = "Org6"
    elif "Org9" in json_path:
        org_name = "Org9"
    else:
        raise ValueError(f"Impossibile determinare l'organizzazione dal file: {json_path}")
        
    print(f"--- Avvio Benchmark per {org_name} ---")
    
    for date_obj in dates:
        date_str = date_obj.strftime('%Y-%m-%d')
        print(f"\n[{date_str}] Inizio elaborazione...")
        
        # 1. Preparazione Fatti Base
        generate_board_baseline_facts(json_path, date_str, "encodings/facts/board_base_facts.lp")
        
        with open("encodings/facts/board_base_facts.lp", 'r') as f:
            if len(f.readlines()) < 10:
                print(f"[{date_str}] Nessun dato sufficiente. Salto.")
                continue
                
        # 2. Predizioni ML Dinamiche
        success, df_today = predict_daily_affinities(json_path, date_str, org_name, "encodings/facts/ml_link_facts.lp")
        if not success: continue

        # --- ESECUZIONE BOARD ---
        print(" -> Risoluzione Board (Baseline vs ML)...")
        b_ass_base, t_b_base, cost_b_base = run_board_solver(
            "encodings/board_rules.lp", "encodings/facts/board_base_facts.lp", None, timeout
        )
        b_ass_ml, t_b_ml, cost_b_ml = run_board_solver(
            "encodings/ml_board_rules.lp", "encodings/facts/board_base_facts.lp", "encodings/facts/ml_link_facts.lp", timeout
        )
        
        # --- ESECUZIONE AGENDA ---
        print(" -> Validazione Fisica in Agenda...")
        generate_agenda_facts(b_ass_base, json_path, date_str, "encodings/facts/agenda_base_facts.lp")
        t_a_base, cost_a_base, dropped_base = run_agenda_solver("encodings/facts/agenda_base_facts.lp", timeout=30.0)
        
        generate_agenda_facts(b_ass_ml, json_path, date_str, "encodings/facts/agenda_ml_facts.lp")
        t_a_ml, cost_a_ml, dropped_ml = run_agenda_solver("encodings/facts/agenda_ml_facts.lp", timeout=30.0)

        print("\n   [ANALISI CONTINUITÀ]")
        print("   Baseline:")
        analizza_continuita_assistenziale(b_ass_base, df_today)
        print("   Machine Learning:")
        analizza_continuita_assistenziale(b_ass_ml, df_today)
        
        # --- METRICHE ---
        terra_board_base = sum(1 for a in b_ass_base if a['operator_id'] == -1)
        terra_board_ml = sum(1 for a in b_ass_ml if a['operator_id'] == -1)
        
        df_base_ass = pd.DataFrame(b_ass_base)
        df_ml_ass = pd.DataFrame(b_ass_ml)
        
        if not df_base_ass.empty and not df_ml_ass.empty:
            merged = df_base_ass.merge(df_ml_ass, on='patient_id', suffixes=('_base', '_ml'))
            spostati = (merged['operator_id_base'] != merged['operator_id_ml']).sum()
        else:
            spostati = 0
        
        results.append({
            'Data': date_str,
            'Pazienti_Totali': len(b_ass_base),
            'Tempo_Board_Base_s': round(t_b_base, 3),
            'Tempo_Board_ML_s': round(t_b_ml, 3),
            'Tempo_Agenda_Base_s': round(t_a_base, 3),
            'Tempo_Agenda_ML_s': round(t_a_ml, 3),
            'Pazienti_Spostati_ML': spostati,
            'Terra_Board_Base': terra_board_base,
            'Terra_Board_ML': terra_board_ml,
            'Terra_Agenda_Base': dropped_base,
            'Terra_Agenda_ML': dropped_ml
        })
        
        print(f"[{date_str}] Finito! Tempo ML: {t_b_ml + t_a_ml:.2f}s | Sessioni fallite in Agenda: {dropped_ml}")

        # AVVIO DEL PLOT
        parse_and_plot_agenda("encodings/facts/agenda_base_facts.lp", "encodings/agenda_rules.lp", "Agenda BASELine")
        parse_and_plot_agenda("encodings/facts/agenda_ml_facts.lp", "encodings/agenda_rules.lp", "Agenda MACHINE LEARNING")

    df_results = pd.DataFrame(results)
    df_results.to_csv(f"results/final_benchmark_{org_name}.csv", index=False)
    return df_results

data_url_9 = 'data/anon_boardHistory_Org9_2023-01-01_2023-01-31.json'
df_final_9 = run_benchmark(data_url_9, start_date='2023-01-02', end_date='2023-01-31', timeout=20.0)
display(df_final_9)

'''data_url_6 = 'data/anon_boardHistory_Org6_2023-01-01_2023-01-31.json'
df_final_6 = run_benchmark(data_url_6, start_date='2023-01-02', end_date='2023-01-4', timeout=20.0)
display(df_final_6)'''